# 🔴 Solution: Implement BatchNorm（参考版）

## 📋 题目描述

**难度：** Medium

实现带训练/推理模式的批归一化。

BatchNorm 在训练时使用批统计量、推理时使用运行统计量对每个特征进行归一化，从而稳定深度网络训练。

**签名:** `my_batch_norm(x, gamma, beta, running_mean, running_var, eps=1e-5, momentum=0.1, training=True) -> Tensor`

**参数:**
- `x` — 输入张量 (N, D)
- `gamma`, `beta` — 可学习的仿射参数 (D,)
- `running_mean`, `running_var` — 运行统计量 (D,)，训练时原地更新

**返回:** 归一化并仿射变换后的张量，形状与 x 相同

**约束:**
- 训练模式：使用批统计量，用 momentum 更新运行统计量
- 推理模式：仅使用运行统计量
- 批方差使用 `unbiased=False`

**提示：** 训练：`mean/var = x.mean/var(dim=0)`，用 `momentum` 更新运行统计量
推理：直接使用 `running_mean/running_var`
两者：`gamma * (x - mean) / sqrt(var + eps) + beta`


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def batchnorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, running_mean: torch.Tensor, running_var: torch.Tensor, training: bool = True, momentum: float = 0.1, eps: float = 1e-5) -> torch.Tensor:
    # Batch Norm 沿 batch 维度归一化（对每个特征通道独立计算）
    # x shape: [batch, features] 或 [batch, features, ...]
    # 归一化维度：除 batch 维外的所有维度

    # ---- 确定归一化维度 ----
    # 对于 2D: [B, F] → dims=[0]
    # 对于 4D: [B, C, H, W] → dims=[0, 2, 3]
    # 即"除特征维外的所有维度"
    dims = [0] + list(range(2, x.ndim))

    if training:
        # ---- 训练模式：用当前 batch 的统计量 ----
        # Step 1: 计算 batch 均值
        mean = x.mean(dim=dims)

        # Step 2: 计算 batch 方差
        var = x.var(dim=dims, unbiased=False)

        # Step 3: 更新 running statistics（指数移动平均）
        # running_mean = (1-momentum) * running_mean + momentum * batch_mean
        # momentum 越大，越信任当前 batch
        running_mean[:] = (1 - momentum) * running_mean + momentum * mean
        running_var[:] = (1 - momentum) * running_var + momentum * var
    else:
        # ---- 推理模式：用训练时积累的 running statistics ----
        mean = running_mean
        var = running_var

    # ---- Step 4: 归一化 ----
    # reshape mean/var 为可广播的形状
    # 2D: [F] → [1, F]; 4D: [C] → [1, C, 1, 1]
    shape = [1, -1] + [1] * (x.ndim - 2)
    mean_r = mean.view(shape)
    var_r = var.view(shape)

    x_norm = (x - mean_r) / torch.sqrt(var_r + eps)

    # ---- Step 5: 仿射变换 ----
    return weight.view(shape) * x_norm + bias.view(shape)

In [ ]:
# Verify
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training behavior: normalize with batch stats and update running stats
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Column means:", out_train.mean(dim=0))
print("[Train] Column stds: ", out_train.std(dim=0))
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference behavior: use running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Column means (using running stats):", out_eval.mean(dim=0))

## 📝 核心思路总结

1. **训练 vs 推理**：训练用 batch 统计，推理用 running EMA
2. **归一化维度**：沿 batch + 空间维，每个特征通道独立
3. **running statistics**：指数移动平均，momentum 控制更新速度
4. **view 广播**：reshape 统计量为可广播形状是关键